<a href="https://colab.research.google.com/github/mayaelnaggar/W26-MP1-Mohamed-Abdelmoniem-Maya-ElNaggar-T09/blob/main/Regression_Mini_Project_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Mining — Linear Regression — Mini-Project 1

**Course:** [CSEN911] Data Mining (Winter 2026)  
**Instructor:** Dr. Ayman Alserafi  
**Due:** 17 October 2026, 11:59 PM

**Dataset:** `energy_data.csv`  



Name: Maya ElNaggar

ID: 61-9282

Tutorial: T9 BI

---

Name: Mohamed Abdelmoniem

ID: 61-1408

Tutorial: T9 BI

---


The dataset contains building-level energy readings and contextual attributes.

Each row represents a building observation. Columns include:

<div style="font-size:20px;">

| **Column** | **Description** |
|-------------|-----------------|
| **Building_ID** | Unique identifier for each building record. Used to distinguish one building entry from another. |
| **Building_Type** | Category describing the primary use of the building (e.g., Residential, Commercial, Industrial, Educational, etc.). |
| **Governorate** | The administrative region (governorate) where the building is located (e.g., Cairo, Giza, Alexandria). |
| **Neighborhood** | The smaller district or local area within the governorate where the building is located. |
| **Day_of_Week** | The day on which the energy consumption measurement was recorded (e.g., Sunday, Monday, etc.). |
| **Occupancy_Level** | The relative number of occupants or activity level in the building, typically categorized as *Low*, *Medium*, or *High*. |
| **Appliances_Usage_Level** | Indicates how intensively appliances are used in the building *Low*, *Medium*, or *High*. |
| **SquareFootage** | The total floor area of the building (numeric). Serves as a proxy for building size, often influencing energy usage. |
| **Last_Maintenance_Date** | The date of the last maintainance done on the building. |
| **Average_Temperature** | The average ambient temperature (in °C) recorded during the data period. |
| **Energy_Consumption** | The total energy used by the building, typically measured in kilowatt-hours (kWh).|

</div>


## Importing Libraries & Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    df = pd.read_csv('energy_data.csv')
except:
    df = pd.read_csv('https://raw.githubusercontent.com/GUC-DM/W2026/refs/heads/main/data/energy_data.csv')

df.head()

,Building_ID,Building_Type,Governorate,Neighborhood,Day_of_Week,Occupancy_Level,Appliances_Usage_Level,SquareFootage,Last_Maintenance_Date,Average_Temperature,Energy_Consumption
0,BLD-1000-UD,Residential,Alexandria,Smouha,WeDnesday,High,Low,7063m2,2020-01-01,28.61,2713.95 kWh
1,BLD-1001-AX,Commercial,Giza,+Mohandessin14,tuesDAY,High,High,44372m2,2022-02-24,NaN,5744.99 kWh
2,BLD-1002-IH,Industrial,Cairo,New Cairo,SunDay,Medium,Low,19255,2021-02-22,37.88,4101.24 kWh
3,BLD-1003-HE,NaN,NaN,+92Dokki,TuesDay,Low,High,13265,2023-07-30,35.06,3009.14 kWh
4,BLD-1004-XD,Commercial,Alexandria,Smouha,Monday,Low,Low,13375,2022-08-12,28.82,3279.17 kWh


## Data Inspection

Perform data inspection tasks here (recommended for data understanding).

The first step we performed is running "df.tail()" to verify that the issues seen at the head of the data are systematic issues rather than one off issues; The issues that were found were the following
-  Null values in column Building_Type
-  Null values in column Governate
-  Null values in column Average_Temperature
- Inconsistent Units in SquareFootage (Some values have m2 and some don't)
- Inconsistent formatting in column Neighborhood (Stray symbols & random numbers)
- Incorret capitalization in column Day_of_Week
- Energy_Consumption carries a kWh suffix when it should only contain the number


In [ ]:

df.tail()


,Building_ID,Building_Type,Governorate,Neighborhood,Day_of_Week,Occupancy_Level,Appliances_Usage_Level,SquareFootage,Last_Maintenance_Date,Average_Temperature,Energy_Consumption
1095,BLD-2095-OH,Commercial,Giza,(Dokki50,Saturday,High,Low,1161m2,2022-04-21,27.85,3010.81 kWh
1096,BLD-2096-RH,Residential,NaN,Dokki-71,Sunday,Medium,Medium,37943m2,2024-10-31,36.23,4248.49 kWh
1097,BLD-2097-JZ,Commercial,Giza,_Mohandessin91,SunDay,Low,Medium,1558,2021-04-18,20.00,2843.6 kWh
1098,BLD-2098-ZP,Industrial,Alexandria,*23Smouha,saturday,Medium,Low,2145,2023-09-14,34.43,3348.39 kWh
1099,BLD-2099-GL,Residential,Cairo,-New Cairo52,friDAY,High,Medium,42414,2020-12-09,40.37,4722.59 kWh


Then we used "df.info()" checks the structure and data types included in a dataset.
In our dataset, there are multiple inconsistencies such as:
- Null values: 227 missing values in Governorate, 110 missing values in Building_Type and 110 missing values in Average_Temperature
- Inconsistent formatting in column Neighborhood
- Incorret capitalization in column Days_of_the_Week
- Incorrect data types:
    - Last_Maintenance_Date should be "datetime64" instead of "object"
    - SquareFootage should include numeric values only without "m2"
    - Energy_Consumption is object when it should be numeric.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Building_ID             1100 non-null   object 
 1   Building_Type           990 non-null    object 
 2   Governorate             873 non-null    object 
 3   Neighborhood            1100 non-null   object 
 4   Day_of_Week             1100 non-null   object 
 5   Occupancy_Level         1100 non-null   object 
 6   Appliances_Usage_Level  1100 non-null   object 
 7   SquareFootage           1100 non-null   object 
 8   Last_Maintenance_Date   1100 non-null   object 
 9   Average_Temperature     990 non-null    float64
 10  Energy_Consumption      1100 non-null   object 
dtypes: float64(1), object(10)
memory usage: 94.7+ KB


In the previous step, we found the missing data values per column. We needed to find the exact rows because it can tell us if a single row is missing one or more data values to determine if it is worth keeping or not. Since rows with one missing value is easy to fix but those with more than 1 might not be salvageable, we then used "df.loc[df.isna().sum(axis=1)>1]" to find rows with more than 1 missing gap. As a result, we found around 47 rows, which is around 4% of the data that had more than 1 missing section in the row

In [ ]:
df.loc[df.isna().sum(axis=1)>1]

,Building_ID,Building_Type,Governorate,Neighborhood,Day_of_Week,Occupancy_Level,Appliances_Usage_Level,SquareFootage,Last_Maintenance_Date,Average_Temperature,Energy_Consumption
3,BLD-1003-HE,NaN,NaN,+92Dokki,TuesDay,Low,High,13265,2023-07-30,35.06,3009.14 kWh
34,BLD-1034-LS,NaN,NaN,Dokki,Tuesday,Low,Medium,3800,2020-01-01,26.93,2205.2 kWh
60,BLD-1060-IC,NaN,NaN,*98Gleem,FriDay,Low,High,7559m2,2022-05-01,32.35,4027.56 kWh
129,BLD-1129-AS,Commercial,NaN,Smouha@32,Saturday,Medium,Low,12061,2020-01-01,NaN,3010.22 kWh
146,BLD-1146-DD,NaN,NaN,@Heliopolis16,SaturDay,Medium,High,26529m2,2022-11-09,35.85,5143.54 kWh
149,BLD-1149-NO,NaN,Giza,Dokki@41,SaTurday,Low,Low,18086m2,2020-01-01,NaN,2685.54 kWh
157,BLD-1157-GR,NaN,Cairo,%49New Cairo,WEDNESDAY,High,Low,21177m2,2021-08-06,NaN,4276.15 kWh
201,BLD-1201-GV,NaN,NaN,Smouha!20,Sunday,High,High,27245,2021-12-21,45.99,5474.72 kWh
210,BLD-1210-SD,Residential,NaN,(92Gleem,SuNday,Medium,Low,10416,2020-01-01,NaN,2512.1 kWh
223,BLD-1223-VD,NaN,NaN,+90Maadi,FRIday,Low,High,45047m2,2024-12-31,37.91,5750.07 kWh


We used "value_counts()" on each categorical column to list every unique value and how often it appears. This shows all the spelling inconsistencies that "info()" could not show earlier.
Results:
  - Building_Type and Governorate each contain all 3 categories and their counts match the missing values found earlier (990 and 873 non-null)
  - Day_of_Week has 42 distinct values instead of 7 because each day is written with different variations of capitalizations. According to this data, Saturday (279) and Sunday (270) are more common than the rest of the days, which are around 100 to 120 each
  - Neighborhood has 848 distinct values. The real names, such as Gleem, Smouha and Dokki, appear with random symbols so one neighborhood is split into many different categories
  - Last_Maintenance_Date has 727 different values, but "2024-12-31" and "2020-01-01" account for about 15% of the data, with 89 and 83 rows respectively. Every other date appears at most 4 times
  - Occupancy_Level and Appliance_Usage_Level are clean with no missing values and containing only Low, Medium and High

In [ ]:
for col in ['Building_Type' , 'Governorate', 'Neighborhood', 'Day_of_Week', 'Occupancy_Level' , 'Appliances_Usage_Level', 'Last_Maintenance_Date']:
  print(df[col].value_counts(), '\n')

Building_Type
Residential    349
Commercial     325
Industrial     316
Name: count, dtype: int64 

Governorate
Alexandria    310
Giza          293
Cairo         270
Name: count, dtype: int64 

Neighborhood
Smouha          45
Gleem           44
Dokki           41
Mohandessin     38
New Cairo       32
                ..
%Smouha42        1
+Maadi7          1
^Dokki97         1
=59Dokki         1
%51New Cairo     1
Name: count, Length: 848, dtype: int64 

Day_of_Week
Sunday       54
Saturday     53
saturDAY     49
SuNday       48
SATURDAY     47
SaTurday     47
SunDay       47
sunDAY       44
SaturDay     44
SUNDAY       39
saturday     39
sunday       38
FRIday       27
wednesday    22
Wednesday    22
THURSDAY     22
tuesDAY      22
WednesDay    22
tuesday      21
WEDNESDAY    21
MonDay       21
wednESDAY    20
TuesDay      20
Monday       19
thursDAY     19
MoNday       19
FriDay       18
ThursDay     18
friDAY       18
MONDAY       17
TUESDAY      17
monDAY       16
monday       16
Frid

We used "df.describe(include'all')" to get a summary of every column.
we found that Data is complete with no duplicate records due to Building_Id not being repeated and being present for every record. the categorical counts allign with all our previous findings SquareFootage and Energy_Consumption must be converted from text to numbers. Average_Temperature also contained a minimum value of -4.91 celsius which is far below the rest of the data while being unrealistic for cairo and would cause the data to skew.

In [ ]:
df.describe(include='all')

,Building_ID,Building_Type,Governorate,Neighborhood,Day_of_Week,Occupancy_Level,Appliances_Usage_Level,SquareFootage,Last_Maintenance_Date,Average_Temperature,Energy_Consumption
count,1100,990,873,1100,1100,1100,1100,1100,1100,990.000000,1100
unique,1100,3,3,848,42,3,3,1095,727,NaN,1100
top,BLD-2099-GL,Residential,Alexandria,Smouha,Sunday,High,Low,49653m2,2024-12-31,NaN,4722.59 kWh
freq,1,349,310,45,54,368,381,2,89,NaN,1
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.499404,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.703806,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.910000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.422500,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.260000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.970000,NaN


## Data Pre-Processing & Cleaning

_Apply any data preprocessing and/or feature engineering below. Show/output the changes to the dataset._

## Exploratory Data Analysis

**Q1:** What are the most popular neighborhoods? plot all and order them on the graph (mention top 3)

**Visualization**

**Answer for Q1:** _Your answer here_

**Q2:** Show the distribution of the energy consumption of each Building type.

Which type have the widest distribution of energy consumption?

Which (on average) has the highest consumption?.

**Visualization**

**Answer for Q2:** _Your answer here_

**Q3:** How does the building size affect energy consumption?

**Visualization**

**Answer for Q3:** _Your answer here_

**Q4:** Do buildings consume more energy if not maintained frequently?

**Visualization**

**Answer for Q4:** _Your answer here_

**Q5:** Are all the numerical variables normally distributed, or is there any skewness?

**Visualization**

**Answer for Q5:** _Your answer here_

**Q6:** What is multicollinearity? And why is it a problem for linear regression? Does this problem exist in this
dataset?

**Visualization**

**Answer for Q6:** _Your answer here_

## Data Preparation for Modelling

_Apply any additional data preparation steps needed before modelling below. Show/output the changes to the dataset._

## Modelling

_Apply the linear regression model below._

## Model Evaluation

Evaluate the model you applied.

## Conclusion and Recommendations

Comment on the model performance and your findings from model evaluation. State the problems (if any) and suggest possible solutions. Would you recommend this model for an electrcity company aiming to estimate the energy levels of each building?

**Answer**: your answer here.